In [ ]:
# 1) Install dependencies, then restart runtime once
# Run this cell once to install dependencies and restart the runtime.
# After restart, run this cell again and then continue to the next cell.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_ai_challenge_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime now. After restart, run this cell again, then continue.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue to the next cell.")


In [ ]:
# 2) Setup + data unzip
from google.colab import drive

drive.mount('/content/drive')

import ast
import gc
import glob
import os
import random
import re
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, Subset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
OUTPUT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_multitask_v1"
SUBMIT_PATH = "/content/outputs/submission.csv"

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SUBMIT_PATH), exist_ok=True)

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
SEED = 42
VALID_RATIO = 0.1
SMOKE_TEST = False

# Training can use all tasks; exact-match evaluation uses ORDER only.
TRAIN_TASKS = ("ORDER", "PAIRWISE", "RANK")
EVAL_TASKS = ("ORDER",)

# Use None for full validation. For a quick check, use a number like 50.
BEST_CHECKPOINT_EVAL_LIMIT = None

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("data:", DATA_DIR)
print("output:", OUTPUT_DIR)
print("train tasks:", TRAIN_TASKS)


In [ ]:
# 3) Dataset, collator, prediction helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)


class FrameOrderMultiTaskDataset(Dataset):
    def __init__(self, dataframe, image_root, tasks=("ORDER",), augment=True, seed=42):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.tasks = tuple(tasks)
        self.augment = augment
        self.seed = seed

    def __len__(self):
        return len(self.dataframe) * len(self.tasks)

    def __getitem__(self, index):
        task_count = len(self.tasks)
        row_index = index // task_count
        task = self.tasks[index % task_count]

        row = self.dataframe.iloc[row_index]
        sample_id = str(row["Id"])
        sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])

        image_paths = [
            os.path.join(self.image_root, sample_id, str(row[f"Input_{i}"]))
            for i in range(1, 5)
        ]
        temporal_ranks = [int(value) for value in row["Answer_list"]]

        local_rng = random if self.augment else random.Random(self.seed + index)
        if self.augment:
            permutation = list(range(4))
            local_rng.shuffle(permutation)
            image_paths = [image_paths[i] for i in permutation]
            temporal_ranks = [temporal_ranks[i] for i in permutation]

        if task == "ORDER":
            image_labels = [f"Input image {i}" for i in range(1, 5)]
            instruction = (
                f'The video is described as: "{sentence}"\n\n'
                "The four images are shuffled frames from the video. "
                "Compare the visual states and temporal progression.\n"
                "Return the actual temporal rank of Input image 1, "
                "Input image 2, Input image 3, and Input image 4 "
                "in that order.\n"
                "Respond only with one Python-style permutation list."
            )
            target = str([int(value) for value in temporal_ranks])

        elif task == "PAIRWISE":
            first_index, second_index = local_rng.sample(range(4), 2)
            image_paths = [image_paths[first_index], image_paths[second_index]]
            image_labels = ["Image A", "Image B"]
            instruction = (
                f'The video is described as: "{sentence}"\n\n'
                "Determine which of the two frames occurs earlier in the original video.\n"
                "Respond only with A or B."
            )
            target = "A" if temporal_ranks[first_index] < temporal_ranks[second_index] else "B"

        elif task == "RANK":
            selected_index = local_rng.randrange(4)
            image_labels = [f"Input image {i}" for i in range(1, 5)]
            instruction = (
                f'The video is described as: "{sentence}"\n\n'
                "The four images are shuffled frames from the video. "
                f"What is the actual temporal rank of Input image {selected_index + 1}?\n"
                "Respond only with one integer from 1 through 4."
            )
            target = str(int(temporal_ranks[selected_index]))

        else:
            raise ValueError(f"Unsupported task: {task}")

        return {
            "Id": sample_id,
            "task": task,
            "image_paths": image_paths,
            "image_labels": image_labels,
            "instruction": instruction,
            "target": target,
        }


training_dataset = FrameOrderMultiTaskDataset(training_df, TRAIN_IMAGE_DIR, tasks=TRAIN_TASKS, augment=True, seed=SEED)
validation_dataset = FrameOrderMultiTaskDataset(validation_df, TRAIN_IMAGE_DIR, tasks=EVAL_TASKS, augment=False, seed=SEED)


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def make_messages(example):
    content = []
    for label in example["image_labels"]:
        content.append({"type": "text", "text": f"\n{label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]


def find_last_subsequence(sequence, pattern):
    for start in range(len(sequence) - len(pattern), -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start
    return -1


class QwenFrameOrderCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_prefix_ids = processor.tokenizer.encode(
            "<|im_start|>assistant\n",
            add_special_tokens=False,
        )

    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("Use batch size 1 with this collator.")

        example = examples[0]
        images = [load_rgb(path) for path in example["image_paths"]]
        messages = make_messages(example) + [{"role": "assistant", "content": example["target"]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

        model_inputs = self.processor(text=[text], images=images, padding=False, return_tensors="pt")
        input_ids = model_inputs["input_ids"][0].tolist()
        assistant_pos = find_last_subsequence(input_ids, self.assistant_prefix_ids)

        if assistant_pos >= 0:
            answer_start = assistant_pos + len(self.assistant_prefix_ids)
        else:
            prompt_text = self.processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True)
            prompt_inputs = self.processor(text=[prompt_text], images=images, padding=False, return_tensors="pt")
            answer_start = prompt_inputs["input_ids"].shape[1]

        labels = model_inputs["input_ids"].clone()
        labels[:, :answer_start] = -100
        if "attention_mask" in model_inputs:
            labels[model_inputs["attention_mask"] == 0] = -100
        model_inputs["labels"] = labels
        return model_inputs


def parse_order_output(output_text):
    match = re.search(r"\[\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*,\s*[1-4]\s*\]", output_text)
    if match is None:
        return None
    try:
        result = ast.literal_eval(match.group())
    except (ValueError, SyntaxError, TypeError):
        return None
    if isinstance(result, list) and len(result) == 4 and sorted(result) == [1, 2, 3, 4]:
        return [int(value) for value in result]
    return None


@torch.no_grad()
def predict_order_example(example, max_new_tokens=32):
    model.eval()
    images = [load_rgb(path) for path in example["image_paths"]]
    text = processor.apply_chat_template(make_messages(example), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=images, padding=False, return_tensors="pt").to(model.device)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return output_text, parse_order_output(output_text)


def evaluate_order_exact(dataset, limit=None, desc="Validation"):
    rows = []
    n = len(dataset) if limit is None else min(limit, len(dataset))
    for index in tqdm(range(n), desc=desc):
        example = dataset[index]
        raw_output, prediction = predict_order_example(example)
        answer = [int(value) for value in ast.literal_eval(example["target"])]
        rows.append({
            "Id": example["Id"],
            "Raw_output": raw_output,
            "Prediction": str(prediction) if prediction is not None else None,
            "Answer": str(answer),
            "Correct": prediction == answer,
        })
    return pd.DataFrame(rows)


def parse_order(value):
    if value is None or pd.isna(value):
        return None
    try:
        parsed = value if isinstance(value, list) else ast.literal_eval(str(value))
    except (ValueError, SyntaxError):
        return None
    if not isinstance(parsed, list) or len(parsed) != 4:
        return None
    try:
        parsed = [int(x) for x in parsed]
    except (TypeError, ValueError):
        return None
    return parsed if sorted(parsed) == [1, 2, 3, 4] else None


print("training examples:", len(training_dataset), "validation ORDER examples:", len(validation_dataset), "test rows:", len(test_df))
print("training rows:", len(training_df), "validation rows:", len(validation_df))
print("example tasks:", [training_dataset[i]["task"] for i in range(min(6, len(training_dataset)))])


In [ ]:
# 4) Train
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(
    model,
    LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    ),
)

for name, parameter in model.named_parameters():
    if "visual" in name.lower() or "vision" in name.lower():
        parameter.requires_grad = False

trainable_parameter_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
print("Trainable parameter module count:", len(trainable_parameter_names))
for name in trainable_parameter_names[:30]:
    print(name)
model.print_trainable_parameters()

data_collator = QwenFrameOrderCollator(processor)

if SMOKE_TEST:
    train_data = Subset(training_dataset, range(min(30, len(training_dataset))))
    eval_data = Subset(validation_dataset, range(min(10, len(validation_dataset))))
    max_steps = 100
    eval_steps = 5
else:
    train_data = training_dataset
    eval_data = validation_dataset
    max_steps = -1
    eval_steps = 100

training_argument_values = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 1,
    "max_steps": max_steps,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 1e-4,
    "warmup_ratio": 0.03,
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "eval_steps": eval_steps,
    "logging_steps": 20,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": None,
    "report_to": "none",
    "remove_unused_columns": False,
    "dataloader_num_workers": 0,
    "seed": SEED,
    "data_seed": SEED,
}

if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
    training_argument_values["eval_strategy"] = "steps"
else:
    training_argument_values["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_argument_values)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)


In [ ]:
# 5) Eval + best checkpoint by exact True
# This cell also works when the Train cell was skipped.
if "processor" not in globals():
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )

if "bnb_config" not in globals():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def find_adapter_dirs(output_dir):
    candidates = []
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        candidates.append(output_dir)
    candidates.extend(sorted(glob.glob(os.path.join(output_dir, "checkpoint-*"))))
    return [path for path in candidates if os.path.exists(os.path.join(path, "adapter_config.json"))]


def load_model_for_adapter(adapter_dir):
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    loaded_model = PeftModel.from_pretrained(base_model, adapter_dir)
    loaded_model.eval()
    loaded_model.config.use_cache = True
    return loaded_model


checkpoint_dirs = find_adapter_dirs(OUTPUT_DIR)
print("Checkpoint count to evaluate:", len(checkpoint_dirs))

if "model" not in globals():
    if not checkpoint_dirs:
        raise RuntimeError(f"No model in memory and no adapter checkpoints found in {OUTPUT_DIR}")
    print("No model in memory. Loading first checkpoint for initial eval:", checkpoint_dirs[0])
    model = load_model_for_adapter(checkpoint_dirs[0])

model.eval()
model.config.use_cache = True

validation_result_df = evaluate_order_exact(validation_dataset, desc="Current model")
exact_match_accuracy = validation_result_df["Correct"].mean()
parse_failure_count = validation_result_df["Prediction"].isna().sum()

print(f"Exact-match accuracy: {exact_match_accuracy:.2%}")
print(f"Correct count: {validation_result_df['Correct'].sum()}/{len(validation_result_df)}")
print(f"Parse failure count: {parse_failure_count}")

identity_answer = [1, 2, 3, 4]
identity_baseline_accuracy = validation_result_df["Answer"].apply(lambda text: ast.literal_eval(text) == identity_answer).mean()
print(f"Always [1,2,3,4] baseline: {identity_baseline_accuracy:.2%}")
print(f"Current model exact-match: {exact_match_accuracy:.2%}")

frame_columns = ["Input_1", "Input_2", "Input_3", "Input_4"]
train_ids = set(training_df["Id"].astype(str))
validation_ids = set(validation_df["Id"].astype(str))
print("Overlapping Id count:", len(train_ids & validation_ids))

def frame_signature(row):
    return tuple(sorted(str(row[column]) for column in frame_columns))

train_signatures = set(training_df.apply(frame_signature, axis=1))
validation_signatures = set(validation_df.apply(frame_signature, axis=1))
print("Overlapping frame-group count:", len(train_signatures & validation_signatures))

result_with_group = validation_result_df.copy()
if "No_ordering" in validation_df.columns:
    group_source = validation_df[["Id", "No_ordering"]].copy()
    group_source["Id"] = group_source["Id"].astype(str)
    result_with_group["Id"] = result_with_group["Id"].astype(str)
    result_with_group = result_with_group.merge(group_source, on="Id", how="left", validate="one_to_one")

    def calculate_row_metrics(row):
        prediction = parse_order(row["Prediction"])
        answer = parse_order(row["Answer"])
        if prediction is None or answer is None:
            return pd.Series({"Position_correct": 0, "Position_total": 4, "Exact_correct": 0, "Parse_failed": 1})
        return pd.Series({
            "Position_correct": sum(pred == target for pred, target in zip(prediction, answer)),
            "Position_total": 4,
            "Exact_correct": int(prediction == answer),
            "Parse_failed": 0,
        })

    metric_columns = result_with_group.apply(calculate_row_metrics, axis=1)
    result_with_group = pd.concat([result_with_group, metric_columns], axis=1)
    group_accuracy = result_with_group.groupby("No_ordering").agg(
        sample_count=("Id", "count"),
        correct_positions=("Position_correct", "sum"),
        total_positions=("Position_total", "sum"),
        exact_correct=("Exact_correct", "sum"),
        parse_failures=("Parse_failed", "sum"),
    )
    group_accuracy["position_accuracy"] = group_accuracy["correct_positions"] / group_accuracy["total_positions"]
    group_accuracy["exact_match_accuracy"] = group_accuracy["exact_correct"] / group_accuracy["sample_count"]
    display(group_accuracy)
else:
    print("No_ordering column not found; skipping group evaluation.")

display(validation_result_df.head())


checkpoint_rows = []

best_checkpoint_dir = None
best_checkpoint_accuracy = -1.0

for checkpoint_dir in checkpoint_dirs:
    print("evaluating:", checkpoint_dir)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    model = load_model_for_adapter(checkpoint_dir)

    checkpoint_df = evaluate_order_exact(
        validation_dataset,
        limit=BEST_CHECKPOINT_EVAL_LIMIT,
        desc=os.path.basename(checkpoint_dir),
    )
    accuracy = checkpoint_df["Correct"].mean()
    correct_count = int(checkpoint_df["Correct"].sum())
    total_count = len(checkpoint_df)
    parse_failures = int(checkpoint_df["Prediction"].isna().sum())

    checkpoint_rows.append({
        "checkpoint": checkpoint_dir,
        "exact_match_accuracy": accuracy,
        "correct": correct_count,
        "total": total_count,
        "parse_failures": parse_failures,
    })

    if accuracy > best_checkpoint_accuracy:
        best_checkpoint_accuracy = accuracy
        best_checkpoint_dir = checkpoint_dir

if checkpoint_rows:
    checkpoint_result_df = pd.DataFrame(checkpoint_rows).sort_values(
        ["exact_match_accuracy", "correct"],
        ascending=False,
    ).reset_index(drop=True)

    display(checkpoint_result_df)
    print("BEST_CHECKPOINT_DIR:", best_checkpoint_dir)
    print(f"BEST exact-match: {best_checkpoint_accuracy:.2%}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    model = load_model_for_adapter(best_checkpoint_dir)
else:
    checkpoint_result_df = pd.DataFrame()
    print("No adapter checkpoint found; keeping the current model.")


In [ ]:
# 6) Inference + submission with best checkpoint loaded above
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    image_paths = [
        os.path.join(TEST_IMAGE_DIR, str(row["Id"]), str(row[f"Input_{i}"]))
        for i in range(1, 5)
    ]
    example = {
        "Id": str(row["Id"]),
        "task": "ORDER",
        "image_paths": image_paths,
        "image_labels": [f"Input image {i}" for i in range(1, 5)],
        "instruction": (
            f'The video is described as: "{sentence}"\n\n'
            "The four images are shuffled frames from the video. "
            "Compare the visual states and temporal progression.\n"
            "Return the actual temporal rank of Input image 1, "
            "Input image 2, Input image 3, and Input image 4 "
            "in that order.\n"
            "Respond only with one Python-style permutation list."
        ),
    }

    _, prediction = predict_order_example(example)
    predictions.append({
        "Id": str(row["Id"]),
        "Answer": str(prediction if prediction is not None else [1, 2, 3, 4]),
    })

submission_df = pd.DataFrame(predictions)
submission_df.to_csv(SUBMIT_PATH, index=False)
print("saved:", SUBMIT_PATH)
if "best_checkpoint_dir" in globals():
    print("using checkpoint:", best_checkpoint_dir)
display(submission_df.head())
